# NutriVision AI: Notebook 3 — Transfer Learning & Model Training

**Project:** NutriVision AI — AI-Powered Food Recognition and Nutrition Analysis  
**Author:** NutriVision AI Engineering Team  
**Dataset:** `SohlHealth/enhanced-indian-food-classification` (15,404 images across 43 classes)  
**Target Environment:** Google Colab (with GPU acceleration) / Local Jupyter

---

### Objectives:
1. **Transfer Learning Pipeline:** Fine-tune deep convolutional neural networks pre-trained on ImageNet-1K for 43 Indian food classes.
2. **Architecture Comparison:** Train and compare:
   - **MobileNetV2** (Lightweight inverted residual architecture, ~3.5M parameters, low latency for edge/Streamlit deployment).
   - **ResNet50** (Deep residual architecture, ~25.6M parameters, high feature representation capacity).
3. **Loss & Regularization:** Use class-weighted `CrossEntropyLoss` to handle class imbalance, `AdamW` optimizer with weight decay ($10^{-4}$), and dropout ($0.3$).
4. **Learning Rate Schedule & Mixed Precision:** Implement `CosineAnnealingLR` and FP16 automatic mixed precision (`torch.cuda.amp`) for efficient training.
5. **Early Stopping & Checkpoint Saving:** Track validation loss with patience = 3, saving the optimal model checkpoint as `best_model.pt`.
6. **Training History Export:** Save training curves and metrics to `training_history.json`.


## 1. Environment Setup & GPU Verification

In [ ]:
!pip install --quiet datasets huggingface_hub pyarrow pillow matplotlib seaborn pandas scikit-learn tqdm torch torchvision

import os
import sys
import json
import time
import copy
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
import torchvision.transforms as transforms

# Hardware Check
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch Version: {torch.__version__}")
print(f"Active Device: {device}")
if device.type == "cuda":
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"Available VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")


## 2. Dataset Ingestion & Stratified DataLoaders

In [ ]:
from datasets import load_dataset, concatenate_datasets

DATASET_REPO = "SohlHealth/enhanced-indian-food-classification"
print("Loading dataset from Hugging Face...")
raw_ds = load_dataset(DATASET_REPO)
pooled_dataset = concatenate_datasets([raw_ds[s] for s in raw_ds.keys()])
print(f"Total samples: {len(pooled_dataset):,}")

# Extract labels and classes
all_labels = [item["label"] for item in pooled_dataset]
all_indices = np.arange(len(pooled_dataset))
classes = sorted(list(set(all_labels)))
class_to_idx = {c: i for i, c in enumerate(classes)}
idx_to_class = {i: c for i, c in enumerate(classes)}
num_classes = len(classes)

# Stratified Split: 70% Train, 15% Val, 15% Test
SEED = 42
train_idx, temp_idx, train_y, temp_y = train_test_split(
    all_indices, all_labels, test_size=0.30, stratify=all_labels, random_state=SEED
)
val_idx, test_idx, val_y, test_y = train_test_split(
    temp_idx, temp_y, test_size=0.50, stratify=temp_y, random_state=SEED
)

print(f"Train samples: {len(train_idx):,} | Val: {len(val_idx):,} | Test: {len(test_idx):,}")

# Compute class weights for weighted loss
train_counts = Counter(train_y)
class_weights = [len(train_idx) / (num_classes * train_counts[c]) for c in classes]
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)

# Transforms
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class IndianFoodDataset(Dataset):
    def __init__(self, dataset, indices, class_to_idx, transform=None):
        self.dataset = dataset
        self.indices = indices
        self.class_to_idx = class_to_idx
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        real_idx = int(self.indices[idx])
        sample = self.dataset[real_idx]
        image = sample["image"]
        if image.mode != "RGB":
            image = image.convert("RGB")
        label = self.class_to_idx[sample["label"]]
        if self.transform:
            image = self.transform(image)
        return image, label

BATCH_SIZE = 32
NUM_WORKERS = 2 if device.type == "cuda" else 0

train_dataset = IndianFoodDataset(pooled_dataset, train_idx, class_to_idx, train_transform)
val_dataset = IndianFoodDataset(pooled_dataset, val_idx, class_to_idx, val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda"))
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda"))

print(f"DataLoaders successfully initialized. Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")


## 3. Model Architecture Factory (MobileNetV2 & ResNet50)

We construct a modular model factory that replaces the 1,000-class ImageNet classification head with a custom dropout-regularized linear projection head for 43 Indian food classes.


In [ ]:
def build_model(architecture="mobilenet_v2", num_classes=43, pretrained=True):
    """Construct model with pre-trained backbone and custom 43-class head."""
    arch = architecture.lower()
    
    if arch == "mobilenet_v2":
        weights = models.MobileNet_V2_Weights.DEFAULT if pretrained else None
        model = models.mobilenet_v2(weights=weights)
        in_features = model.classifier[1].in_features
        model.classifier = nn.Sequential(
            nn.Dropout(p=0.3),
            nn.Linear(in_features, num_classes)
        )
    elif arch == "resnet50":
        weights = models.ResNet50_Weights.DEFAULT if pretrained else None
        model = models.resnet50(weights=weights)
        in_features = model.fc.in_features
        model.fc = nn.Sequential(
            nn.Dropout(p=0.3),
            nn.Linear(in_features, num_classes)
        )
    else:
        raise ValueError(f"Unsupported architecture: {arch}")

    return model

# Test instantiation
m2_test = build_model("mobilenet_v2", num_classes=43, pretrained=False)
r50_test = build_model("resnet50", num_classes=43, pretrained=False)
print("MobileNetV2 parameter count:", sum(p.numel() for p in m2_test.parameters()))
print("ResNet50 parameter count:   ", sum(p.numel() for p in r50_test.parameters()))


## 4. Training Engine, Validation & Early Stopping

In [ ]:
class EarlyStopping:
    """Early stops the training if validation loss doesn't improve after a given patience."""
    def __init__(self, patience=3, min_delta=0.001):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = None
        self.early_stop = False

    def __call__(self, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
        elif val_loss > self.best_loss - self.min_delta:
            self.counter += 1
            print(f"EarlyStopping counter: {self.counter} out of {self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_loss = val_loss
            self.counter = 0


def train_one_epoch(model, dataloader, criterion, optimizer, scaler, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, targets in tqdm(dataloader, desc="Training", leave=False):
        images, targets = images.to(device), targets.to(device)
        optimizer.zero_grad()

        with torch.amp.autocast(device_type="cuda" if device.type == "cuda" else "cpu", enabled=(device.type == "cuda")):
            outputs = model(images)
            loss = criterion(outputs, targets)

        if scaler:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == targets).sum().item()
        total += targets.size(0)

    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc


def evaluate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct_top1 = 0
    correct_top5 = 0
    total = 0

    with torch.no_grad():
        for images, targets in tqdm(dataloader, desc="Validating", leave=False):
            images, targets = images.to(device), targets.to(device)

            with torch.amp.autocast(device_type="cuda" if device.type == "cuda" else "cpu", enabled=(device.type == "cuda")):
                outputs = model(images)
                loss = criterion(outputs, targets)

            running_loss += loss.item() * images.size(0)
            
            # Top-1 accuracy
            _, preds = torch.max(outputs, 1)
            correct_top1 += (preds == targets).sum().item()
            
            # Top-5 accuracy
            _, top5_preds = outputs.topk(min(5, outputs.size(1)), dim=1, largest=True, sorted=True)
            correct_top5 += top5_preds.eq(targets.view(-1, 1).expand_as(top5_preds)).sum().item()
            
            total += targets.size(0)

    val_loss = running_loss / total
    val_top1 = correct_top1 / total
    val_top5 = correct_top5 / total
    return val_loss, val_top1, val_top5


## 5. Complete Model Training & Logging Loop

In [ ]:
def run_training_experiment(architecture="mobilenet_v2", epochs=15, lr=3e-4, patience=3):
    print(f"\n{'='*70}")
    print(f"STARTING EXPERIMENT: {architecture.upper()} ({epochs} Epochs, LR: {lr})")
    print(f"{'='*70}")

    model = build_model(architecture=architecture, num_classes=num_classes, pretrained=True).to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
    scaler = torch.amp.GradScaler("cuda") if device.type == "cuda" else None
    early_stopping = EarlyStopping(patience=patience)

    history = {
        "train_loss": [], "train_acc": [],
        "val_loss": [], "val_top1": [], "val_top5": [],
        "epoch_times": []
    }

    best_val_loss = float("inf")
    best_model_weights = None

    for epoch in range(1, epochs + 1):
        t0 = time.time()
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, scaler, device)
        val_loss, val_top1, val_top5 = evaluate(model, val_loader, criterion, device)
        epoch_time = time.time() - t0
        scheduler.step()

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_top1"].append(val_top1)
        history["val_top5"].append(val_top5)
        history["epoch_times"].append(epoch_time)

        print(f"Epoch [{epoch:02d}/{epochs:02d}] ({epoch_time:.1f}s) | "
              f"Train Loss: {train_loss:.4f} Acc: {train_acc*100:.2f}% | "
              f"Val Loss: {val_loss:.4f} Top-1: {val_top1*100:.2f}% Top-5: {val_top5*100:.2f}% | "
              f"LR: {scheduler.get_last_lr()[0]:.6f}")

        # Save best weights
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_weights = copy.deepcopy(model.state_dict())
            print(f"  --> Best model checkpoint updated! (Val Loss: {best_val_loss:.4f})")

        early_stopping(val_loss)
        if early_stopping.early_stop:
            print(f"\nEarly stopping triggered after epoch {epoch}!")
            break

    # Restore best weights
    if best_model_weights is not None:
        model.load_state_dict(best_model_weights)

    return model, history


## 6. Model Training & Comparative Evaluation

We train both **MobileNetV2** (lightweight production candidate) and **ResNet50** (deep benchmark) to compare convergence, accuracy, and inference speed.


In [ ]:
EPOCHS = 15
PATIENCE = 3
LR = 3e-4

# 1. Train MobileNetV2
mobilenet_model, mobilenet_history = run_training_experiment(
    architecture="mobilenet_v2",
    epochs=EPOCHS,
    lr=LR,
    patience=PATIENCE
)

# 2. Train ResNet50
resnet_model, resnet_history = run_training_experiment(
    architecture="resnet50",
    epochs=EPOCHS,
    lr=LR,
    patience=PATIENCE
)


## 7. Comparative Training Curves & Performance Summary

In [ ]:
plt.figure(figsize=(15, 5))

# Loss Curves
plt.subplot(1, 2, 1)
plt.plot(mobilenet_history["train_loss"], label="MobileNetV2 Train Loss", color="blue", linestyle="--")
plt.plot(mobilenet_history["val_loss"], label="MobileNetV2 Val Loss", color="blue")
plt.plot(resnet_history["train_loss"], label="ResNet50 Train Loss", color="green", linestyle="--")
plt.plot(resnet_history["val_loss"], label="ResNet50 Val Loss", color="green")
plt.title("Training and Validation Loss Comparison", fontsize=12, fontweight="bold")
plt.xlabel("Epoch")
plt.ylabel("Cross-Entropy Loss")
plt.legend()
plt.grid(True, alpha=0.3)

# Accuracy Curves
plt.subplot(1, 2, 2)
plt.plot(mobilenet_history["val_top1"], label="MobileNetV2 Val Top-1", color="blue")
plt.plot(mobilenet_history["val_top5"], label="MobileNetV2 Val Top-5", color="blue", linestyle=":")
plt.plot(resnet_history["val_top1"], label="ResNet50 Val Top-1", color="green")
plt.plot(resnet_history["val_top5"], label="ResNet50 Val Top-5", color="green", linestyle=":")
plt.title("Validation Top-1 & Top-5 Accuracy Comparison", fontsize=12, fontweight="bold")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Comparative Summary Table
df_comp = pd.DataFrame([
    {
        "Architecture": "MobileNetV2",
        "Best Val Loss": min(mobilenet_history["val_loss"]),
        "Final Val Top-1": mobilenet_history["val_top1"][-1],
        "Final Val Top-5": mobilenet_history["val_top5"][-1],
        "Avg Epoch Time (s)": np.mean(mobilenet_history["epoch_times"]),
        "Total Params": sum(p.numel() for p in mobilenet_model.parameters())
    },
    {
        "Architecture": "ResNet50",
        "Best Val Loss": min(resnet_history["val_loss"]),
        "Final Val Top-1": resnet_history["val_top1"][-1],
        "Final Val Top-5": resnet_history["val_top5"][-1],
        "Avg Epoch Time (s)": np.mean(resnet_history["epoch_times"]),
        "Total Params": sum(p.numel() for p in resnet_model.parameters())
    }
])
print("\n--- ARCHITECTURE COMPARISON SUMMARY ---")
print(df_comp.to_string(index=False))


## 8. Export Best Model Checkpoint & Training Artifacts

In [ ]:
os.makedirs("../models", exist_ok=True)

# Select best overall model based on validation loss
if min(mobilenet_history["val_loss"]) <= min(resnet_history["val_loss"]):
    best_model = mobilenet_model
    best_arch = "mobilenet_v2"
    best_history = mobilenet_history
else:
    best_model = resnet_model
    best_arch = "resnet50"
    best_history = resnet_history

print(f"Selected Best Architecture: {best_arch.upper()}")

checkpoint_payload = {
    "architecture": best_arch,
    "state_dict": best_model.state_dict(),
    "classes": classes,
    "num_classes": num_classes,
    "val_loss": min(best_history["val_loss"]),
    "val_top1": max(best_history["val_top1"]),
    "val_top5": max(best_history["val_top5"]),
    "input_size": [3, 224, 224],
    "normalization": {
        "mean": [0.485, 0.456, 0.406],
        "std": [0.229, 0.224, 0.225]
    }
}

# Save in current directory for Colab /content/ and in ../models/
torch.save(checkpoint_payload, "best_model.pt")
try:
    torch.save(checkpoint_payload, "../models/best_model.pt")
except Exception:
    pass

print(f"Saved best_model.pt ({os.path.getsize('best_model.pt') / 1e6:.2f} MB)")

with open("training_history.json", "w", encoding="utf-8") as f:
    json.dump({
        "selected_architecture": best_arch,
        "mobilenet_history": mobilenet_history,
        "resnet_history": resnet_history
    }, f, indent=2)

try:
    with open("../models/training_history.json", "w", encoding="utf-8") as f:
        json.dump({
            "selected_architecture": best_arch,
            "mobilenet_history": mobilenet_history,
            "resnet_history": resnet_history
        }, f, indent=2)
except Exception:
    pass

print("Saved training_history.json")


In [ ]:
# ==============================================================================
# One-Click Colab Download (Save trained model to your local computer)
# ==============================================================================
try:
    from google.colab import files
    print("Downloading best_model.pt to your PC...")
    files.download("best_model.pt")
    files.download("training_history.json")
    print("Download initiated! Place best_model.pt in D:\DL_CAPSTONE\models\")
except ImportError:
    print("Running in local environment; files saved to disk.")
